In [1]:
import os
import torch
import numpy as np
import pandas as pd
import librosa
from tqdm import tqdm
from transformers import WhisperModel, WhisperFeatureExtractor
from pathlib import Path

# ── 1. CONFIGURACIÓN Y RUTAS ──
PROJECT_ROOT = Path.cwd().parent
DATASET_TSV = PROJECT_ROOT / 'mtg-jamendo-dataset' / 'data' / 'autotagging_moodtheme.tsv'
AUDIO_DIR = PROJECT_ROOT / 'data' / 'audio' 
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'embeddings_fase3'

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── 2. CARGAR DATASET ──
print("Cargando dataset...")
registros = []
with open(DATASET_TSV, 'r', encoding='utf-8') as f:
    next(f)
    for linea in f:
        if not linea.strip(): continue
        columnas = linea.strip().split('\t')
        if len(columnas) >= 6:
            track_id = columnas[0].replace('track_', '').lstrip('0') 
            if track_id == '': track_id = '0'
            registros.append({
                'track_id': track_id,
                'path': columnas[3]
            })

df_completo = pd.DataFrame(registros)
print(f"Total de canciones en índice: {len(df_completo)}")

# ── 3. CONFIGURACIÓN DEL MODELO WHISPER (Rama Semántica) ──
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando dispositivo: {device}")

# Usamos Whisper "base" para que sea rápido y eficiente (dimensión latente: 512)
model_name = "openai/whisper-base"
print("Cargando el Feature Extractor y el modelo Whisper...")
feature_extractor = WhisperFeatureExtractor.from_pretrained(model_name)
model = WhisperModel.from_pretrained(model_name).to(device)
model.eval()

# ── 4. EXTRACCIÓN TEMPORAL Y POOLING ──
whisper_temporal_embeddings = []
track_ids_validos = []

print("Iniciando extracción semántica (Pooling de 1500 -> 30 frames/audio)...")

duracion_total = 30.0
sr = 16000 # Whisper usa estrictamente 16kHz

for idx, row in tqdm(df_completo.iterrows(), total=len(df_completo)):
    track_id = str(row['track_id'])
    
    # Lógica de rutas para fallback a .low.mp3
    path_limpio = str(row['path']).strip() 
    audio_path_normal = AUDIO_DIR / path_limpio
    audio_path_low = AUDIO_DIR / path_limpio.replace('.mp3', '.low.mp3')

    if os.path.exists(audio_path_normal):
        audio_path = audio_path_normal
    elif os.path.exists(audio_path_low):
        audio_path = audio_path_low
    else:
        continue 
        
    try:
        # 1. Cargar exactamente 30 segundos a 16kHz
        audio_array, _ = librosa.load(str(audio_path), sr=sr, duration=duracion_total)
        
        # 2. Rellenar con ceros si el audio dura menos de 30s
        if len(audio_array) < int(sr * duracion_total):
            pad_length = int(sr * duracion_total) - len(audio_array)
            audio_array = np.pad(audio_array, (0, pad_length), mode='constant')
            
        # 3. Whisper Feature Extractor (Calcula el log-Mel de 30s de golpe)
        inputs = feature_extractor(audio_array, sampling_rate=sr, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad():
            # Solo pasamos por el Encoder (no nos interesa que genere texto, solo que entienda el audio)
            encoder_outputs = model.encoder(**inputs)
            
            # Dimensiones de Whisper Base: (1, 1500, 512)
            hidden_states = encoder_outputs.last_hidden_state.cpu().numpy()[0]
            
            # 4. Magia de Alineación Dimensional (Average Pooling)
            # Convertimos (1500, 512) -> (30, 50, 512) y calculamos la media en el bloque de 50
            secuencia_30_frames = hidden_states.reshape(30, 50, -1).mean(axis=1)
            
        # Guardamos la secuencia (Shape final: 30, 512)
        whisper_temporal_embeddings.append(secuencia_30_frames)
        track_ids_validos.append(track_id)
            
    except Exception as e:
        pass # Ignorar audios corruptos

# ── 5. GUARDAR MATRIZ TRIDIMENSIONAL ──
print("\nGuardando resultados temporales de Whisper...")

np.save(OUTPUT_DIR / 'whisper_temporal_embeddings.npy', np.array(whisper_temporal_embeddings))
np.save(OUTPUT_DIR / 'track_ids_whisper_temporal.npy', np.array(track_ids_validos))

print(f"¡Completado! Se extrajeron las secuencias de {len(track_ids_validos)} canciones.")
print(f"Dimensiones de la matriz final: {np.array(whisper_temporal_embeddings).shape}")

c:\Users\Usuario\miniconda3\envs\gpu_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Cargando dataset...
Total de canciones en índice: 18486
Usando dispositivo: cuda
Cargando el Feature Extractor y el modelo Whisper...


c:\Users\Usuario\miniconda3\envs\gpu_env\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Usuario\.cache\huggingface\hub\models--openai--whisper-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular H

Iniciando extracción semántica (Pooling de 1500 -> 30 frames/audio)...


100%|██████████| 18486/18486 [16:55<00:00, 18.20it/s]



Guardando resultados temporales de Whisper...
¡Completado! Se extrajeron las secuencias de 18486 canciones.
Dimensiones de la matriz final: (18486, 30, 512)
